# Steam Game Success Prediction — Exploratory Data Analysis

This notebook performs EDA on the **training set only**, following the corrected project workflow.

**Important:** Validation and test data must not be used to make decisions in this notebook.

In [1]:
# Import the libraries required for EDA

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [2]:
# Define the project paths

PROJECT_ROOT = Path.cwd().parent
TRAIN_FILE = PROJECT_ROOT / "data" / "processed" / "train.csv"
FIGURES_DIR = PROJECT_ROOT / "results" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
print("Training file:", TRAIN_FILE)

Training file: d:\Steam_Games_Success_Prediction\data\processed\train.csv


In [3]:
# Load the training dataset

if not TRAIN_FILE.exists():
    raise FileNotFoundError(
        f"Training dataset not found at: {TRAIN_FILE}\n"
        "Run the cleaning, target construction, and split steps first."
    )

df = pd.read_csv(TRAIN_FILE, low_memory=False)
print("Training dataset loaded successfully.")
print("Shape:", df.shape)

FileNotFoundError: Training dataset not found at: d:\Steam_Games_Success_Prediction\data\processed\train.csv
Run the cleaning, target construction, and split steps first.

## 1. Basic Dataset Overview

Inspect the training data before making feature-engineering decisions.

In [ ]:
# Display the basic structure of the training data

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("\nColumn names:")
print(df.columns.tolist())

In [ ]:
# Display data types, missing values, and unique values

overview = pd.DataFrame({
    "dtype": df.dtypes,
    "missing_count": df.isna().sum(),
    "missing_percentage": df.isna().mean() * 100,
    "unique_values": df.nunique(dropna=True)
})
print(overview)

## 2. Target Distribution

Inspect the already-defined `Success` target. The target definition itself must not be changed to achieve a preferred class balance.

In [ ]:
# Check the distribution of the Success target

TARGET = "Success"
if TARGET not in df.columns:
    raise KeyError("The 'Success' column is not present.")

target_counts = df[TARGET].value_counts(dropna=False)
target_percentages = df[TARGET].value_counts(normalize=True, dropna=False) * 100
target_summary = pd.DataFrame({
    "count": target_counts,
    "percentage": target_percentages
})
print(target_summary)

In [ ]:
# Plot the target distribution

plt.figure(figsize=(6, 4))
df[TARGET].value_counts().sort_index().plot(kind="bar")
plt.title("Success Target Distribution — Training Set")
plt.xlabel("Success")
plt.ylabel("Number of Games")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "target_distribution.png", dpi=300)
plt.show()

## 3. Numeric Feature Analysis

Inspect summary statistics and selected distributions of numeric predictor candidates.

In [ ]:
# Display summary statistics for numeric columns

numeric_columns = df.select_dtypes(include=np.number).columns.tolist()
print(df[numeric_columns].describe().T)

In [ ]:
# Plot distributions of selected numeric variables

candidate_columns = [
    "price", "required_age", "dlc_count",
    "achievements", "positive", "negative"
]
available_columns = [c for c in candidate_columns if c in df.columns]

for column in available_columns:
    plt.figure(figsize=(7, 4))
    df[column].dropna().plot(kind="hist", bins=40)
    plt.title(f"Distribution of {column}")
    plt.xlabel(column)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f"distribution_{column}.png", dpi=300)
    plt.show()

## 4. Categorical Feature Analysis

Inspect the most frequent categories. High-cardinality identifiers and raw free-text fields are not automatically model features.

In [ ]:
# Display the most frequent values of selected categorical columns

categorical_candidates = [
    "windows", "mac", "linux", "developers", "publishers", "genres"
]

for column in categorical_candidates:
    if column in df.columns:
        print(f"\n--- {column} ---")
        print(df[column].value_counts(dropna=False).head(10))

## 5. Correlation Analysis

Correlation is used as an exploratory diagnostic, not as an automatic feature-selection rule.

In [ ]:
# Calculate and display correlations among numeric variables

numeric_df = df.select_dtypes(include=np.number)
if numeric_df.shape[1] >= 2:
    correlation_matrix = numeric_df.corr(numeric_only=True)
    print(correlation_matrix.round(2))
else:
    print("Not enough numeric columns for correlation analysis.")

In [ ]:
# Create a correlation heatmap for numeric variables

if numeric_df.shape[1] >= 2:
    plt.figure(figsize=(12, 9))
    plt.imshow(correlation_matrix, aspect="auto")
    plt.colorbar(label="Correlation")
    plt.xticks(range(len(correlation_matrix.columns)), correlation_matrix.columns, rotation=90)
    plt.yticks(range(len(correlation_matrix.columns)), correlation_matrix.columns)
    plt.title("Numeric Feature Correlation — Training Set")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "numeric_correlation_heatmap.png", dpi=300)
    plt.show()

## 6. EDA Notes

Record observations supported by the training data, such as skewed distributions, missingness, high-cardinality fields, and notable relationships.

In [ ]:
# Save a compact EDA summary for later documentation

eda_summary_file = PROJECT_ROOT / "results" / "metrics" / "eda_summary.txt"
eda_summary_file.parent.mkdir(parents=True, exist_ok=True)

with open(eda_summary_file, "w", encoding="utf-8") as file:
    file.write("STEAM GAME SUCCESS PREDICTION - EDA SUMMARY\n")
    file.write("Missing values:\n")
    file.write(f"Training rows: {df.shape[0]}\n")
    file.write(f"Training columns: {df.shape[1]}\n\n")
            file.write("Missing values:\n")
    file.write(df.isna().sum().sort_values(ascending=False).to_string())
    file.write("\n\nTarget distribution:\n")
    file.write(target_summary.to_string())

print(f"EDA summary saved to: {eda_summary_file}")